# Testscript for Master Thesis 

## TOC 

- [Setup](#setup)
- [Load and Sample Tweets for Labeling](#load_tweets)
- [Define the different prompts](#define_prompts)
- [Run the Labeling Tasks (Single)](#run_labeling_single)
- [Run the Labeling Tasks (Batch)](#run_labeling_batch)

#### Open ToDo's
-Clean tweets from nbsps etc.  
-rewrite data frame creation and data collection into separate functions

In [313]:
# Setup
# cd to path of current script
import os
import sys
os.chdir(sys.path[0])

! git pull

Already up to date.


<a id="setup"></a>
### SANDBOX & SETUP

In [178]:
# Load API key from gitignored api_key module 
from api_key import API_KEY as API_KEY

In [110]:
# base Request
# test capabilties
from openai import OpenAI

# Setup open AI client using api key
client = OpenAI(
  api_key=API_KEY
)

# Test prompt
return_number = "Return me a number between 35 and 47. Only return the number"

# # Test completion
# completion = client.chat.completions.create(
#   model="gpt-4o-mini",
#   store=True,
#   messages=[
#     {"role": "user", 
#      "content": return_number
#     }
#   ],
#   temperature = 2 # default temp = 1  
# )

# print("Response: ", completion.choices[0].message.content);

Response:  43


<a id="load_tweets"></a>
### Load and Sample Tweets for Labeling

#### Load the Tweets from the Kern et al. (2023) paper

In [273]:
# load the full tweets as used by CK
import pandas as pd
# pd.set_option('max_colwidth', 1000)
pd.set_option('display.max_colwidth', 0)

# Load the tweet files
folder_path = 'Tweets_CK/'
train_file = 'full_train_s.csv'
test_file = 'full_test_s.csv'

train_df = pd.read_csv(os.path.join(folder_path, train_file))
test_df = pd.read_csv(os.path.join(folder_path, test_file))

# Reorder test_df columns to match train_df
test_df = test_df[train_df.columns]

# Add the 'original_split' column
train_df['original_split'] = 'train'
test_df['original_split'] = 'test'

# Combine them
tweets_full = pd.concat([train_df, test_df], ignore_index=True)

# Rename columns for a consistent style 
tweets_full = (tweets_full.
               rename(columns={'tweet.id': 'tweet_id', 
                               'batch.tweet': 'batch_tweet',
                               'hate.speech': 'hate_speech', 
                               'offensive.language': 'offensive_language'}))

In [277]:
# Get Basic Info 
num_rows = tweets_full.shape[0]
num_cols = tweets_full.shape[1]
num_unique_tweets = tweets_full['tweet_id'].nunique()
num_batches = tweets_full['batch_tweet'].nunique()
tweet_counts = (tweets_full[['tweet_hashed','batch_tweet']]
                .drop_duplicates().groupby('batch_tweet').size())
average_tweets_per_batch = tweet_counts.mean()

print(f"Number of rows: {num_rows}")
print(f"Number of columns: {num_cols}")
print(f"Number of unique tweet IDs: {num_unique_tweets}")
print(f"Number of Batches: {num_batches}")
print(f'Average Tweets per Batch: {average_tweets_per_batch}')
tweets_full.sort_values('tweet_id').head()

Number of rows: 44900
Number of columns: 8
Number of unique tweet IDs: 3000
Number of Batches: 50
Average Tweets per Batch: 60.0


,tweet_id,id,version,batch_tweet,hate_speech,offensive_language,tweet_hashed,original_split
2927,1,7,B,R1,0.0,1.0,@###### bro that hoe live,train
2930,1,13,A,R1,0.0,1.0,@###### bro that hoe live,train
2932,1,155,A,R1,0.0,1.0,@###### bro that hoe live,train
2918,1,125,E,R1,1.0,0.0,@###### bro that hoe live,train
2919,1,173,E,R1,0.0,1.0,@###### bro that hoe live,train


#### Clean tweets from escape characters etc.

In [ ]:
# TO DO?

#### Randomly Sample N tweets and create the result df out of them

In [278]:
# randomly sample N unique tweets
N_SAMPLES = 3

sampled_tweets = (
    tweets_full
    .drop_duplicates(subset='tweet_hashed')
    .sample(n=N_SAMPLES
            ,random_state=42 # no seed to get different cases during dev
           )
    ['tweet_hashed']
)

sampled_tweets.head()

20613    RT @###### Chile I'm taking greyhound everywhere from now on. Yo hoes and bout to fly me into no building or fly me over the Bermuda &#8230;
13966    @###### nicca u read what i typed u aint slo by a long shot dont play dumb now litebrite lol                                                
2154     @###### shut the fuck up. You guys a mediocre washed basura juice. Sit down and win close games against raiders and jets faggot             
Name: tweet_hashed, dtype: object

<a id="define_prompts"></a>
### Define the different Prompts

In [201]:
# Define the prompts

PROMPT_DICT = {
    # SINGLE TWEET PROMPTING
    ### A: Both OL and HS at the same time, OL first
    "A":"""You are given tweets and yor task is to tell whether they contain Offensive Language (OL) or Hate Speech (HS). 
    1) Return *OL* if the tweet contains Offensive Language
    2) Return *HS* if the tweet contains Hate Speech
    If they contain both OL and HS return *OLHS*, if neither OL nor HS return *none*
    Please return only the parts in between the *asterisks* (without asterisks), do not add any explanations.
    Here is the tweet: {tweet}""", 
    ### B: Both OL and HS at the same time, HS first
    "B":"""You are given tweets and yor task is to tell whether they contain Hate Speech (HS) or Offensive Language (OL). 
    1) Return *HS* if the tweet contains Hate Speech
    2) Return *OL* if the tweet contains Offensive language
    If they contain both HS and OL return *HSOL*, if neither HS nor OL return *none*.
    Please return only the parts in between the *asterisks* (without asterisks), do not add any explanations.
    Here is the tweet: {tweet}""", 
    ### C.OL: Only OL
    "C.OL":"""You are given tweets and yor task is to tell whether they contain Offensive Language (OL). 
    Return *OL* if the tweet contains Offensive language
    If they do not contain OL return *none*.
    Please return only the parts in between the *asterisks* (without asterisks), do not add any explanations.
    Here is the tweet: {tweet}""", 
    ### C.HS: Only HS
    "C.HS":"""You are given tweets and yor task is to tell whether they contain Hate Speech (HS). 
    Return *HS* if the tweet contains Hate Speech
    If they do not contain HS return *none*.
    Please return only the parts in between the *asterisks* (without asterisks), do not add any explanations.
    Here is the tweet: {tweet}""" 
    
    
    # SINGLE TWEET PROMPTING WITH EXPLANATION
    "A_ex":"""You will see a series of tweets. For each tweet, we want you to code if it contains hate speech or contains offensive language.
    We define hate speech as:  Language that is used to expresses hatred towards a targeted group or language intended to be derogatory, to humiliate, or to insult the members of the group.
    However, context matters. If a tweet discusses someone else’s hate speech, it may use hateful terms, but the tweet is not hate speech. 
    We define offensive language as:  Language that is highly offensive to certain individuals or groups but does not meet the requirements of hate speech.
    Some tweets contains both hate speech AND offensive language.
    
    For each tweet, we want you to code whether it contains Offensive Language (OL) or Hate Speech (HS). 
    1) Return *OL* if the tweet contains Offensive Language
    2) Return *HS* if the tweet contains Hate Speech
    If they contain both OL and HS return *OLHS*, if neither OL nor HS return *none*
    Please return only the parts in between the *asterisks* (without asterisks), do not add any explanations.
    Here is the tweet: {tweet}""", 
    ### B: Both OL and HS at the same time, HS first
    "B_ex":"""You will see a series of tweets. For each tweet, we want you to code if it contains hate speech or contains offensive language.
    We define hate speech as:  Language that is used to expresses hatred towards a targeted group or language intended to be derogatory, to humiliate, or to insult the members of the group.
    However, context matters. If a tweet discusses someone else’s hate speech, it may use hateful terms, but the tweet is not hate speech. 
    We define offensive language as:  Language that is highly offensive to certain individuals or groups but does not meet the requirements of hate speech.
    Some tweets contains both hate speech AND offensive language.
    
    For each tweet, we want you to code whether it contains Hate Speech (HS) or Offensive Language (OL). 
    1) Return *HS* if the tweet contains Hate Speech
    2) Return *OL* if the tweet contains Offensive language
    If they contain both HS and OL return *HSOL*, if neither HS nor OL return *none*.
    Please return only the parts in between the *asterisks* (without asterisks), do not add any explanations.
    Here is the tweet: {tweet}""", 
    ### C.OL: Only OL
    "C_ex.OL":"""You will see a series of tweets. For each tweet, we want you to code if it contains contains offensive language.
    We define offensive language as:  Language that is highly offensive to certain individuals or groups.
    
    For each tweet, we want you to code whether it contains Offensive Language (OL).
    Return *OL* if the tweet contains Offensive language
    If they do not contain OL return *none*.
    Please return only the parts in between the *asterisks* (without asterisks), do not add any explanations.
    Here is the tweet: {tweet}""", 
    ### C.HS: Only HS
    "C_ex.HS":"""You will see a series of tweets. For each tweet, we want you to code if it contains contains hate speech.
    We define hate speech as:  Language that is used to expresses hatred towards a targeted group or language intended to be derogatory, to humiliate, or to insult the members of the group.
    
    For each tweet, we want you to code whether it contains Hate Speech (HS).
    Return *HS* if the tweet contains Hate Speech
    If they do not contain HS return *none*.
    Please return only the parts in between the *asterisks* (without asterisks), do not add any explanations.
    Here is the tweet: {tweet}""" 
    

    # BATCH PROMPTING (of 50)
    ### B_A: batches with Both OL and HS at the same time (OL first)
    
    
    ### B_B: batches with Both OL and HS at the same time (HS first)
    
    
    # ,"X":"Say X", 
    # "Y":"Say Y"
}

<a id="run_labeling_single"></a>
### Run the Labeling Tasks (Single Tweet Prompting)

#### Create Result DF

In [279]:
# ISSUE: Loop is executed sequential s.t. we cannot use the acctual query-per-sec quote of the API (3r/s)
# SOL: --> rewrite code s.t. it's parallelized

# SELECT CONDITIONS TO RUN THE LABELING FOR
# the selected tweets are stored in: sampled_tweets

# CONDITIONS  (choose out of: A, B, C.HS, C.OL)

CONDITIONS = ['A', 'B']
# CONDITIONS = ['A', 'B', 'C.HS', 'C.OL', 
#               'A_ex', 'B_ex', 'C_ex.HS', 'C_ex.OL'] 

# Loop through each row and call the API N_RESPONSES times
N_RESPONSES = 3

rows = []
for tweet_id, text in enumerate(sampled_tweets, start=1):
    for condition in CONDITIONS:
        rows.append({
            "tweet_id": tweet_id,
            "tweet": text,
            "condition": condition
        })

# # Create the DataFrame
results = pd.DataFrame(rows)
for i in range(1, N_RESPONSES + 1):
    results[f"response_{i}"] = ""
    
results.head()

,tweet_id,tweet,condition,response_1,response_2,response_3
0,1,RT @###### Chile I'm taking greyhound everywhere from now on. Yo hoes and bout to fly me into no building or fly me over the Bermuda &#8230;,A,,,
1,1,RT @###### Chile I'm taking greyhound everywhere from now on. Yo hoes and bout to fly me into no building or fly me over the Bermuda &#8230;,B,,,
2,2,@###### nicca u read what i typed u aint slo by a long shot dont play dumb now litebrite lol,A,,,
3,2,@###### nicca u read what i typed u aint slo by a long shot dont play dumb now litebrite lol,B,,,
4,3,@###### shut the fuck up. You guys a mediocre washed basura juice. Sit down and win close games against raiders and jets faggot,A,,,


In [204]:
%%time 

# Run the actual Loop for API Calls 

for row in range(0,results.shape[0]):
    
    # Get Tweets and Conditions for the Loop
    tweet_to_rate = results.iloc[row]['tweet']
    condition = results.iloc[row]['condition']
    prompt = PROMPT_DICT[condition].format(tweet=tweet_to_rate)
    
    # generate N_RESPONSES responses
    for response in range(1, N_RESPONSES + 1):
        # get response via API Call
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=1,
            store=True,
        )
        label = resp.choices[0].message.content.strip()
        # print(prompt)
        
        # store the result in the result df 
        # results.loc[[row], f"response_{response}"] = label
        results.at[row, f"response_{response}"] = label

n_prompts = results.shape[0]*N_RESPONSES
print(f"Number of API Calls: {n_prompts}")
results

Number of API Calls: 18
CPU times: user 198 ms, sys: 100 ms, total: 298 ms
Wall time: 9.69 s


,tweet_id,tweet,condition,response_1,response_2,response_3
0,1,RT @###### Chile I'm taking greyhound everywhere from now on. Yo hoes and bout to fly me into no building or fly me over the Bermuda &#8230;,A,none,none,none
1,1,RT @###### Chile I'm taking greyhound everywhere from now on. Yo hoes and bout to fly me into no building or fly me over the Bermuda &#8230;,B,none,none,none
2,2,@###### nicca u read what i typed u aint slo by a long shot dont play dumb now litebrite lol,A,OLHS,OLHS,OLHS
3,2,@###### nicca u read what i typed u aint slo by a long shot dont play dumb now litebrite lol,B,HSOL,HSOL,HSOL
4,3,@###### shut the fuck up. You guys a mediocre washed basura juice. Sit down and win close games against raiders and jets faggot,A,OLHS,OLHS,OLHS
5,3,@###### shut the fuck up. You guys a mediocre washed basura juice. Sit down and win close games against raiders and jets faggot,B,HSOL,HSOL,HSOL


In [163]:
# save result file 
import os 
from datetime import date

output_dir = "Data_Collection/"
today = date.today().strftime("%Y_%m_%d")
suffix = "100t_4c_3r"
filename = f"{today}_gpt_labels_{suffix}.csv"

filepath = os.path.join(output_dir, filename)
results.to_csv(filepath, index=False)

<a id="run_labeling_batch"></a>
### Run the Labeling Tasks (Batch Prompting)

#### Sample Random Batches

In [289]:
tweets_full

,tweet_id,id,version,batch_tweet,hate_speech,offensive_language,tweet_hashed,original_split
0,1449,1069,E,R49,1.0,1.0,"Subtweeting little bitch ""taking the high road""",train
1,1449,1153,E,R49,0.0,1.0,"Subtweeting little bitch ""taking the high road""",train
2,1449,1440,E,R49,0.0,1.0,"Subtweeting little bitch ""taking the high road""",train
3,1449,1014,D,R49,1.0,1.0,"Subtweeting little bitch ""taking the high road""",train
4,1449,1063,D,R49,0.0,1.0,"Subtweeting little bitch ""taking the high road""",train
...,...,...,...,...,...,...,...,...
44895,2987,1315,A,R37,0.0,1.0,@###### yeah hoe!,test
44896,2987,1324,A,R37,0.0,1.0,@###### yeah hoe!,test
44897,2988,1312,A,R38,0.0,1.0,@###### @###### @###### @###### Lets hangout Monday nigs?,test
44898,2988,1315,A,R38,0.0,1.0,@###### @###### @###### @###### Lets hangout Monday nigs?,test


In [298]:
# sample radnom batches
N_BATCH_SAMPLES = 2
sampled_batches = (pd.Series(tweets_full['batch_tweet']
                   .unique())
                   .sample(n=N_BATCH_SAMPLES, random_state=42))

# get the tweets in these batches 
batch_sampled_tweets = (tweets_full[tweets_full['batch_tweet']
                                    .isin(sampled_batches)]
                       [['batch_tweet','tweet_hashed']].drop_duplicates())

print(f'Number of Batches Sampled: {N_BATCH_SAMPLES}')
print(f"Number of Unique Tweets Sampled: {len(batch_sampled_tweets['tweet_hashed'])}")
print(f'Sampled Batch IDs: {sampled_batches.to_list()}')
batch_sampled_tweets.iloc[0:3]

Number of Batches Sampled: 2
Number of Unique Tweets Sampled: 120
Sampled Batch IDs: ['R24', 'R6']


,batch_tweet,tweet_hashed
210,R24,@###### ratchet white trash
959,R6,The family of Ron O'Neal needs to sue Wu-Tang for that trash song they released in his name.
1363,R24,RT @###### Ima catfish a bitch with fake arms


#### Create Results DF

In [299]:
batch_sampled_tweets

,batch_tweet,tweet_hashed
210,R24,@###### ratchet white trash
959,R6,The family of Ron O'Neal needs to sue Wu-Tang for that trash song they released in his name.
1363,R24,RT @###### Ima catfish a bitch with fake arms
1453,R6,RT @###### @###### not only you. and that pom stuff? totally porn.
1557,R24,They some bitch ass niggas
...,...,...
35597,R6,RT @###### &#8220;@###### I just fucked the life out this bitch! &#128514;&#128514;&#128514;&#128164;&#8221;&#128514;&#128514;&#128514;
35660,R6,These hoes be lying to all of us nigga I got news for the monogamous niggas yo bitch fucking anonymous niggas!!!
35711,R6,@###### Squad builders gets you bitches
35780,R24,RT @###### Apartheid. &#8220;@###### pedestrian check = show me your freedom papers you darkies.&#8221;


In [308]:
# ISSUE: Loop is executed sequential s.t. we cannot use the acctual query-per-sec quote of the API (3r/s)
# SOL: --> rewrite code s.t. it's parallelized

# SELECT CONDITIONS TO RUN THE LABELING FOR
# the selected tweets are stored in: sampled_tweets

# CONDITIONS  (choose out of: A, B, C.HS, C.OL)

CONDITIONS = ['A', 'B']
# CONDITIONS = ['A', 'B', 'C.HS', 'C.OL', 
#               'A_ex', 'B_ex', 'C_ex.HS', 'C_ex.OL'] 

# Loop through each row and call the API N_RESPONSES times
N_RESPONSES = 3

rows = []
# Iterate through df rows
for tweet_id, row in enumerate(batch_sampled_tweets.itertuples(index=False), start=1):
    for condition in CONDITIONS:
        rows.append({
            "tweet_id": tweet_id,
            "batch_tweet": row.batch_tweet,   
            "tweet": row.tweet_hashed,
            "condition": condition
        })

# Create the DataFrame
results = pd.DataFrame(rows)

# Add response columns
for i in range(1, N_RESPONSES + 1):
    results[f"response_{i}"] = ""

results.head()

,tweet_id,batch_tweet,tweet,condition,response_1,response_2,response_3
0,1,R24,@###### ratchet white trash,A,,,
1,1,R24,@###### ratchet white trash,B,,,
2,2,R6,The family of Ron O'Neal needs to sue Wu-Tang for that trash song they released in his name.,A,,,
3,2,R6,The family of Ron O'Neal needs to sue Wu-Tang for that trash song they released in his name.,B,,,
4,3,R24,RT @###### Ima catfish a bitch with fake arms,A,,,


# Notes 
1) The tweets can be processed in parts to either (e.g., the first 2k for all conditions)  
A) be able to split the task into free sub parts or   
B) to be able to better estimate the total costs  

2) Probably the Batch Calls have to be done in a separate code snippet

3) Costs: 
4o-mini : 
Input:          15ct  /1M Tokens
Cached Input:   7.5ct /1M Tokens (might be our case)
Output:         60ct  /1M Tokens


In [136]:
# Get Token Counts 
from utils import count_tokens
import statistics
# count tokens for gpt-3.5-turbo model (default in tiktoken package 
print(count_tokens(PROMPT_DICT['A'])) # 111 TOKENS for standard condition prompt 
print(statistics.fmean([count_tokens(x) for x in sampled_tweets])) # 15 Average Tweet Token length
# response has max 2 tokens

111
15.8
